# LA listings cleaning: executed validation

Policy approved by Christos is recorded in **Decisions made** in plan.md. Raw files are read-only. This notebook independently reloads source/output data, checks preservation and eligibility, summarizes audit evidence, and regenerates all four outputs into a temporary directory to compare bytes.

No rows are dropped; missing values are not imputed. Extreme prices remain in medians. Demand proxies exclude possible inactivity. License classifications describe format rather than verified compliance. Install requirements.txt before rerunning.


In [1]:
import hashlib
import json
import subprocess
import sys
import tempfile
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / 'scripts/clean_listings.py').exists():
    root = root.parent
assert (root / 'scripts/clean_listings.py').exists()
report = json.loads((root / 'data/clean/validation_report.json').read_text(encoding='utf-8'))
clean_tokens = pd.read_csv(root / 'data/clean/listings_clean.csv', dtype='string', keep_default_na=False)
sample = pd.read_csv(root / 'data/clean/listings_sample.csv', dtype='string', keep_default_na=False)
raw = pd.concat([
    pd.read_csv(root / f'data/raw/listings_{date}.csv', dtype='string', keep_default_na=False).assign(snapshot_date=date)
    for date in ['2026-03-16', '2026-06-15']
], ignore_index=True)
assert len(raw) == len(clean_tokens) == 89839
assert not clean_tokens.duplicated(['id','snapshot_date']).any()
assert 0 < len(sample) <=50 and not sample.duplicated(['id','snapshot_date']).any()
print('Raw/clean rows:', len(raw), '/', len(clean_tokens), '; sample:', len(sample))
print('Every stage retained all rows; drops:', report['dropped_rows'])


Raw/clean rows: 89839 / 89839 ; sample: 50
Every stage retained all rows; drops: 0


## Input preservation, safe keys, and unchanged source measures

These checks compare raw tokens and numeric amounts against independently reloaded outputs, including missingness, unmodified license, category labels, capacity, estimates, review rates, and t/f mappings.


In [2]:
for c in ['id','host_id','scrape_id','snapshot_date','source','room_type','property_type',
          'neighbourhood_cleansed','neighbourhood_group_cleansed','license']:
    assert raw[c].tolist() == clean_tokens[c].tolist(), c
assert clean_tokens.id.str.fullmatch(r'\d+').all()
assert clean_tokens.host_id.str.fullmatch(r'\d+').all()
print('19-digit listing IDs:', int(clean_tokens.id.str.len().eq(19).sum()))
assert raw.price.tolist() == clean_tokens.price_raw.tolist()
for c in ['price','price_quote_price_per_night','minimum_nights','maximum_nights','accommodates',
          'bathrooms','bedrooms','beds','estimated_occupancy_l365d','estimated_revenue_l365d','reviews_per_month']:
    original = raw[c].replace('', pd.NA)
    if c == 'price':
        original = original.str.replace('$','',regex=False).str.replace(',','',regex=False)
    expected = pd.to_numeric(original, errors='raise').astype('Float64')
    actual = pd.to_numeric(clean_tokens[c].replace('',pd.NA), errors='raise').astype('Float64')
    pd.testing.assert_series_equal(expected, actual, check_names=False)
for c in ['last_scraped','calendar_last_scraped','first_review','last_review',
          'price_quote_checkin_date','price_quote_checkout_date']:
    assert raw[c].tolist() == clean_tokens[c].tolist()
for c in ['host_is_superhost','host_has_profile_pic','host_identity_verified','has_availability']:
    assert raw[c].replace({'t':'True','f':'False'}).tolist() == clean_tokens[c].tolist()
assert int(clean_tokens.price.eq('').sum()) ==14475
assert int(clean_tokens.minimum_nights.eq('').sum()) ==18
assert int(clean_tokens.reviews_per_month.eq('').sum()) ==23776
for c in ['host_response_rate','host_acceptance_rate','instant_bookable','host_since']:
    assert raw[c].eq('').all()
print('PASS: keys, raw license, categories, price amounts, capacity, dates, rates, estimates and Boolean maps preserved as documented.')


19-digit listing IDs: 42805
PASS: keys, raw license, categories, price amounts, capacity, dates, rates, estimates and Boolean maps preserved as documented.


## Stage accounting and source-specific price missingness

Changed-row counts refer to existing source-cell representations; added fields and omitted columns are accounted for separately in validation_report.json. A row can change in several stages or carry several flags, so these totals are not additive.


In [3]:
stages = pd.DataFrame([{'step':x['step'],'input_rows':x['input_rows'],
    'output_rows':x['output_rows'],'drops':x['dropped_rows'],
    'existing_rows_changed':x['distinct_existing_rows_changed'],
    'changed_cells':sum(v['changed_rows'] for v in x['changed_columns'].values()),
    'added_columns':len(x['added_columns']),'omitted_columns':len(x['removed_columns'])}
    for x in report['steps']])
assert stages.input_rows.eq(stages.output_rows).all() and stages.drops.eq(0).all()
print(stages.to_string(index=False))
print('\nPrice missingness by source, before eligibility:')
print(pd.DataFrame(report['price_missingness_by_source_before_eligibility']).to_string(index=False))
print('\nPrice/quote overlap:')
print(pd.DataFrame(report['price_quote_overlap']).T.to_string())


                                                       step  input_rows  output_rows  drops  existing_rows_changed  changed_cells  added_columns  omitted_columns
            1. Raw string ingestion and filename provenance       89839        89839      0                      0              0              3                0
                 2. Blank-only missing-token interpretation       89839        89839      0                  89839        1535003              0                0
                       3. Select approved analytical schema       89839        89839      0                      0              0              0               27
  4. Parse amounts, counts, ISO dates and nullable Booleans       89839        89839      0                  89837         603658              1                0
        5. Flag suspicious values and classify stay cohorts       89839        89839      0                      0              0             17                0
        6. License status an

## Cohorts, flags, license handling, and eligibility

Primary pricing uses short, priced, city-scrape rows. Possible inactivity affects demand eligibility only. Low/high prices remain unchanged and enter medians when otherwise eligible. Blank/N/A licenses are classified missing but remain exact raw strings.


In [4]:
number = lambda c: pd.to_numeric(clean_tokens[c].replace('',pd.NA)).astype('Float64')
minimum, price = number('minimum_nights'), number('price')
expected_stay = pd.Series('unknown', index=clean_tokens.index, dtype='string')
expected_stay.loc[minimum.lt(30).fillna(False)]='short'
expected_stay.loc[minimum.ge(30).fillna(False)]='long'
assert clean_tokens.stay_type.equals(expected_stay)
inactive = number('number_of_reviews_ltm').eq(0) & number('availability_365').ge(300)
primary = expected_stay.eq('short') & price.notna() & clean_tokens.source.eq('city scrape')
demand = expected_stay.eq('short') & clean_tokens.source.eq('city scrape') & ~inactive
assert clean_tokens.primary_pricing_eligible.eq('True').equals(primary)
assert clean_tokens.demand_eligible.eq('True').equals(demand)
assert not (clean_tokens.demand_eligible.eq('True') & inactive).any()
assert clean_tokens.no_reviews.eq('True').tolist() == number('number_of_reviews').eq(0).tolist()
licenses = clean_tokens.license
assert clean_tokens.loc[licenses.eq('') | licenses.str.casefold().eq('n/a'),'license_status'].eq('missing').all()
norm = clean_tokens.license_normalized.ne('')
assert clean_tokens.loc[norm,'license_normalized'].tolist() == licenses.loc[norm].str.upper().tolist()
assert not clean_tokens.flag_impossible_value.eq('True').any()
print(pd.crosstab(clean_tokens.snapshot_date,clean_tokens.stay_type).to_string())
rows = []
for date, x in report['by_snapshot'].items():
    rows.append({'snapshot':date,'pricing_eligible':x['flag_counts']['primary_pricing_eligible']['true'],
        'demand_eligible':x['flag_counts']['demand_eligible']['true'],
        'possible_inactivity':x['flag_counts']['flag_possible_inactivity']['true'],
        'price_below_10':x['flag_counts']['flag_price_low']['true'],
        'price_above_5000':x['flag_counts']['flag_price_high']['true'],
        'distinct_flagged_rows':x['distinct_rows_with_any_flag'],
        'multiple_flags':x['distinct_rows_with_multiple_flags']})
print('\n'+pd.DataFrame(rows).to_string(index=False))
print('\nLicense status:\n'+pd.crosstab(clean_tokens.snapshot_date,clean_tokens.license_status).to_string())


stay_type       long  short  unknown
snapshot_date                       
2026-03-16     22586  23495        7
2026-06-15     20481  23259       11

  snapshot  pricing_eligible  demand_eligible  possible_inactivity  price_below_10  price_above_5000  distinct_flagged_rows  multiple_flags
2026-03-16             19096            17687                 8323              25               124                  36808           18232
2026-06-15             20550            18527                 8913              20               143                  38017           15421

License status:
license_status  exempt  hsr_registered  missing  other_identifier
snapshot_date                                                    
2026-03-16        3670            7109    32992              2317
2026-06-15        3867            7061    30559              2264


## Segment coverage, medians, and difficult-case sample

Coverage denominators include unpriced supply. Segment keys preserve source neighbourhood and room labels. The report includes all segment medians and long/unknown cohort summaries. Demand summaries are review/calendar proxies and explicitly labeled modeled occupancy.


In [5]:
keys = ['snapshot_date','neighbourhood_cleansed','room_type']
view=clean_tokens.assign(price=price)
primary_supply=view.loc[view.stay_type.eq('short') & view.source.eq('city scrape')]
expected=primary_supply.groupby(keys,sort=True).agg(eligible_supply_rows=('id','size'),priced_rows=('price','count'),median_price=('price','median'))
reported=pd.DataFrame(report['segment_summaries']['primary_pricing_segments']).astype({c:'string' for c in keys}).set_index(keys).sort_index()
pd.testing.assert_frame_equal(expected.astype('Float64'),reported[expected.columns].astype('Float64'),check_names=False)
assert ((reported.priced_coverage>=0) & (reported.priced_coverage<=1)).all()
assert (reported.priced_coverage-reported.priced_rows/reported.eligible_supply_rows).abs().max()<1e-9
print('Primary pricing segments:',len(reported))
print(reported.head(12).to_string())
sample_keys=pd.MultiIndex.from_frame(sample[keys[:1]+['id']])
clean_keys=pd.MultiIndex.from_frame(clean_tokens[keys[:1]+['id']])
assert sample_keys.isin(clean_keys).all() and sample.reason.ne('').all()
coverage=pd.DataFrame(report['sample']['case_coverage']).T
assert (coverage.loc[coverage.available>0,'sampled']>0).all()
print('\nDifficult-case coverage:\n'+coverage.to_string())
print('\nSample reasons:\n'+sample[['id','snapshot_date','reason']].head(10).to_string(index=False))


Primary pricing segments: 1056
                                                      eligible_supply_rows  median_price  priced_coverage  priced_rows
snapshot_date neighbourhood_cleansed room_type                                                                        
2026-03-16    Acton                  Entire home/apt                    13       295.000         1.000000           13
                                     Private room                        3       210.000         1.000000            3
              Adams-Normandie        Entire home/apt                     6       349.290         1.000000            6
                                     Private room                        2       168.500         1.000000            2
              Agoura Hills           Entire home/apt                    24       450.515         1.000000           24
                                     Private room                        6       122.375         1.000000            6
              Agu

## Fresh regeneration and unchanged raw files

This cell runs the script into a fresh temporary output directory and compares SHA-256 of every approved artifact with the delivered outputs. It also verifies both input hashes against the report. Temporary artifacts are removed automatically; raw files are never written.


In [6]:
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()
artifacts=['listings_clean.csv','listings_sample.csv','validation_report.json','data_dictionary.md']
assert sorted(p.name for p in (root/'data/clean').iterdir())==sorted(artifacts)
with tempfile.TemporaryDirectory(prefix='la_notebook_validation_') as temp:
    result=subprocess.run([sys.executable,str(root/'scripts/clean_listings.py'),'--output-dir',temp],
        cwd=root,capture_output=True,text=True,check=True)
    for name in artifacts:
        assert sha256(root/'data/clean'/name)==sha256(Path(temp)/name),name
        print('Identical:',name,sha256(root/'data/clean'/name))
for date, x in report['inputs'].items():
    assert sha256(root/'data/raw'/x['source_file'])==x['sha256']
print('PASS: all four outputs regenerated byte-for-byte; both raw hashes unchanged.')


Identical: listings_clean.csv 52f600b7f7f4b5d3ff8866136d71688ce9832214be6f673bee0efcc9d1eeab76
Identical: listings_sample.csv 0c00550cb0cc0b0dea822bd9ba802b7b0a88a691d9b971c650f4509df746fc87
Identical: validation_report.json 4e01b1941d2ee10fd409db7a66dacff7bd53e048464a459f3a0f73dbe904e3df
Identical: data_dictionary.md d144729c5114fa68895bf332b19e09af58d34bb03b43b176f9b4e3f57133eef9
PASS: all four outputs regenerated byte-for-byte; both raw hashes unchanged.
